In [2]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Adiciona a pasta raiz do projeto no path
ROOT = Path.cwd().parents[0]  # notebooks/ -> projeto/
sys.path.append(str(ROOT))

from src.config import DATA_DIR
from src.io import load_data
from src.cleaning import normalize_columns, basic_null_report, clean_currency, clean_sales_df
from src.viz import plot_numeric_hist


In [3]:
df = load_data(DATA_DIR / "chocolate_sales_2.csv")
df = clean_sales_df(df)
df.head()


,sales_person,country,product,date,amount,boxes_shipped
0,Jehu Rudeforth,UK,Mint Chip Choco,04/01/2022,5320.0,180
1,Van Tuxwell,India,85% Dark Bars,01/08/2022,7896.0,94
2,Gigi Bohling,India,Peanut Butter Cubes,07/07/2022,4501.0,91
3,Jan Morforth,Australia,Peanut Butter Cubes,27/04/2022,12726.0,342
4,Jehu Rudeforth,UK,Peanut Butter Cubes,24/02/2022,13685.0,184


In [15]:
# KPIs principais
total_sales_value = df["amount"].sum().round(0)
total_countries = df["country"].nunique()
total_products = df["product"].nunique()
count_sales_persons = df["sales_person"].nunique()
count_sales = len(df)
total_sales_by_country = df.groupby("country")["amount"].sum().sort_values(ascending=False)

# Gráfico de colunas: total de vendas por país
ax = total_sales_by_country.sort_values(ascending=False).plot(kind="bar", figsize=(12, 6))

# Adicionar os valores no topo das colunas
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",        # formato do número
        label_type="edge"  # coloca no topo
    )


# carrega o valor total de vendas
fig, ax = plt.subplots(figsize=(6, 3))

ax.text(
    0.5, 0.6,
    f"{total_sales_value:,.0f}",
    ha="center",
    va="center",
    fontsize=36,
    fontweight="bold"
)

ax.text(
    0.5, 0.35,
    "Total de Vendas",
    ha="center",
    va="center",
    fontsize=14,
    color="gray"
)

# Remove eixos
ax.axis("off")

# Fundo tipo card
fig.patch.set_facecolor("white")
ax.set_facecolor("#f5f5f5")

plt.show()


In [ ]:
# Relatório de valores nulos
basic_null_report(df)

In [ ]:
# Vendas por produto (top 15)
total_sales_by_product = df.groupby("product")["amount"].sum().sort_values(ascending=False).head(15)
ax = total_sales_by_product.plot(kind="barh", figsize=(10, 6))
ax.set_xlabel("Total de vendas ($)")
ax.set_title("Top 15 produtos por faturamento")
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 vendedores por faturamento
sales_by_person = df.groupby("sales_person")["amount"].sum().sort_values(ascending=False).head(10)
ax = sales_by_person.plot(kind="barh", figsize=(10, 5))
ax.set_xlabel("Total de vendas ($)")
ax.set_title("Top 10 vendedores por faturamento")
plt.tight_layout()
plt.show()

In [ ]:
# Série temporal: vendas ao longo do tempo (requer data em datetime)
df_date = df.copy()
df_date["date"] = pd.to_datetime(df_date["date"], format="%d/%m/%Y", errors="coerce")
monthly = df_date.dropna(subset=["date"]).groupby(df_date["date"].dt.to_period("M"))["amount"].sum()
ax = monthly.plot(kind="line", figsize=(12, 4), marker="o")
ax.set_xlabel("Mês")
ax.set_ylabel("Total de vendas ($)")
ax.set_title("Evolução das vendas por mês")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Histograma do valor das vendas (amount)
plot_numeric_hist(df, "amount", bins=40)